In [49]:
import pandas as pd

station_df = pd.read_csv("../data/processed/feature_engineered_ev_dataset.csv")

area_df = pd.read_csv("../data/external/area_coordinates.csv")

In [50]:
station_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2229 entries, 0 to 2228
Data columns (total 61 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   address                              2229 non-null   str    
 1   latitude                             2229 non-null   float64
 2   longitude                            2229 non-null   float64
 3   staff                                2229 non-null   int64  
 4   postal_code                          2229 non-null   int64  
 5   available                            2229 non-null   float64
 6   capacity                             2229 non-null   float64
 7   cost_per_unit                        2229 non-null   float64
 8   total                                2229 non-null   float64
 9   supports_upi                         2229 non-null   int64  
 10  supports_cash                        2229 non-null   int64  
 11  supports_card                        2229

In [51]:
station_df[["latitude", "longitude"]].drop_duplicates().shape[0]

1747

In [52]:
station_df.shape

(2229, 61)

In [54]:
station_df.loc[
    (station_df["latitude"] < 20) |
    (station_df["longitude"] < 20),
    ["latitude", "longitude","address"]
]

,latitude,longitude,address
855,0.000000,0.000000,near Bablu dairy Khadar Madanpur delhi
1235,0.000000,0.000000,A 35 Shalimar Bagh delhi 110088
1236,0.000000,0.000000,A 35 Shalimar Bagh delhi 110088
1279,0.043945,0.000000,a1240 Main Services Road Gamdi
1285,0.000000,0.000000,Sadh Char Pusta Som Bazar
...,...,...,...
1923,17.441831,78.504351,Secunderabad
1924,17.441835,78.504356,Secunderabad
1928,17.441840,78.504357,Secunderabad
1931,17.441883,78.504386,Secunderabad


In [55]:
non_delhi = station_df[
    (station_df["latitude"] < 28) |
    (station_df["latitude"] > 29) |
    (station_df["longitude"] < 76) |
    (station_df["longitude"] > 78)
]

print("Non-Delhi rows:", len(non_delhi))

Non-Delhi rows: 140


In [56]:
station_df = station_df[
    station_df["latitude"].between(28, 29) &
    station_df["longitude"].between(76, 78)
].reset_index(drop=True)

In [58]:
station_df.to_csv(
    "../data/processed/feature_engineered_ev_dataset_2.csv",
    index=False
)

In [79]:
station_coord = station_df[["latitude", "longitude","address"]].drop_duplicates().reset_index(drop=True)

station_coord.head()

,latitude,longitude,address
0,28.568238,77.219666,"NDSE Grid, BRPL South Extension"
1,28.541995,77.260583,Scada office kalka ji
2,28.571189,77.259806,Ashram Chowk Mathura Road
3,28.588991,77.253240,Nizamuddin Railway station
4,28.549427,77.254636,"BSES Bhawan, Nehru Place, New Delhi 110048"


In [80]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):

    R = 6371

    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat/2)**2 +
        cos(lat1)*cos(lat2)*sin(dlon/2)**2
    )

    c = 2*atan2(sqrt(a), sqrt(1-a))

    return R*c

In [81]:
distance_matrix = station_coord.copy()

In [82]:
for _, area in area_df.iterrows():

    area_name = area["Area"]

    area_lat = area["Latitude"]

    area_lon = area["Longitude"]

    distance_matrix[area_name] = distance_matrix.apply(
        lambda row: haversine(
            row["latitude"],
            row["longitude"],
            area_lat,
            area_lon
        ),
        axis=1
    )

In [83]:
distance_matrix.head()

,latitude,longitude,address,Vasant Kunj,Greater Kailash,Janakpuri,Punjabi Bagh,Rohini,Noida Sector 18,IGI Airport,...,AIIMS,Pitampura,Connaught Place,Rajouri Garden,Kalkaji,Preet Vihar,Karol Bagh,Lajpat Nagar,Shahdara,Civil Lines
0,28.568238,77.219666,"NDSE Grid, BRPL South Extension",8.049425,2.884738,14.189872,14.475909,25.675967,10.388013,11.763267,...,0.950985,16.631142,7.040371,12.605772,4.379965,10.988023,9.717365,2.289276,13.539508,12.060547
1,28.541995,77.260583,Scada office kalka ji,10.592181,2.253020,19.067824,19.303487,30.471224,7.147027,15.764195,...,5.680019,21.273827,10.835640,17.541511,0.798153,11.569153,13.986879,3.329476,14.905346,15.365171
2,28.571189,77.259806,Ashram Chowk Mathura Road,11.565376,3.266536,17.714450,17.010752,28.038873,6.464266,15.695550,...,4.884031,18.696576,7.917301,15.607352,2.469260,8.547698,11.215541,1.676929,11.749477,12.203566
3,28.588991,77.253240,Nizamuddin Railway station,12.056910,4.731594,16.559979,15.290940,26.202594,7.387050,15.402109,...,4.868144,16.813623,5.921625,14.116131,4.479823,7.132094,9.271368,2.566167,10.045018,10.124057
4,28.549427,77.254636,"BSES Bhawan, Nehru Place, New Delhi 110048",10.213978,1.560727,18.174275,18.303204,29.466184,7.363762,15.122226,...,4.786455,20.263841,9.849100,16.569267,0.409478,10.977332,12.982739,2.323330,14.219856,14.433872


In [84]:
area_columns = area_df["Area"].tolist()

In [85]:
distance_matrix["Nearest_Area"] = (
    distance_matrix[area_columns]
    .idxmin(axis=1)
)

In [86]:
distance_matrix["Nearest_Distance_km"] = (
    distance_matrix[area_columns]
    .min(axis=1)
)

In [87]:
distance_matrix.head()

,latitude,longitude,address,Vasant Kunj,Greater Kailash,Janakpuri,Punjabi Bagh,Rohini,Noida Sector 18,IGI Airport,...,Connaught Place,Rajouri Garden,Kalkaji,Preet Vihar,Karol Bagh,Lajpat Nagar,Shahdara,Civil Lines,Nearest_Area,Nearest_Distance_km
0,28.568238,77.219666,"NDSE Grid, BRPL South Extension",8.049425,2.884738,14.189872,14.475909,25.675967,10.388013,11.763267,...,7.040371,12.605772,4.379965,10.988023,9.717365,2.289276,13.539508,12.060547,AIIMS,0.950985
1,28.541995,77.260583,Scada office kalka ji,10.592181,2.253020,19.067824,19.303487,30.471224,7.147027,15.764195,...,10.835640,17.541511,0.798153,11.569153,13.986879,3.329476,14.905346,15.365171,Kalkaji,0.798153
2,28.571189,77.259806,Ashram Chowk Mathura Road,11.565376,3.266536,17.714450,17.010752,28.038873,6.464266,15.695550,...,7.917301,15.607352,2.469260,8.547698,11.215541,1.676929,11.749477,12.203566,Lajpat Nagar,1.676929
3,28.588991,77.253240,Nizamuddin Railway station,12.056910,4.731594,16.559979,15.290940,26.202594,7.387050,15.402109,...,5.921625,14.116131,4.479823,7.132094,9.271368,2.566167,10.045018,10.124057,Lajpat Nagar,2.566167
4,28.549427,77.254636,"BSES Bhawan, Nehru Place, New Delhi 110048",10.213978,1.560727,18.174275,18.303204,29.466184,7.363762,15.122226,...,9.849100,16.569267,0.409478,10.977332,12.982739,2.323330,14.219856,14.433872,Nehru Place,0.325855


In [88]:
distance_matrix["Nearest_Distance_km"].describe()

count    1659.000000
mean        3.286728
std         2.170149
min         0.054085
25%         1.714493
50%         3.022831
75%         4.247215
max        13.218623
Name: Nearest_Distance_km, dtype: float64

In [89]:
distance_matrix.loc[
    distance_matrix["Nearest_Distance_km"].idxmax()
]

latitude                                                       28.821693
longitude                                                      77.170189
address                houseno44/21/2village Bakhtawerpur our Delhi 1...
Vasant Kunj                                                     33.08409
Greater Kailash                                                31.115002
Janakpuri                                                      23.624067
Punjabi Bagh                                                   17.508359
Rohini                                                         13.688507
Noida Sector 18                                                31.768879
IGI Airport                                                    30.305013
Chandni Chowk                                                  19.312713
Mayur Vihar                                                    26.525904
Okhla                                                          33.870967
Model Town                                         

In [91]:
distance_matrix.nlargest(
    10,
    "Nearest_Distance_km"
)[
    [
        "latitude",
        "longitude",
        "address",
        "Nearest_Area",
        "Nearest_Distance_km"
    ]
]

,latitude,longitude,address,Nearest_Area,Nearest_Distance_km
854,28.821693,77.170189,houseno44/21/2village Bakhtawerpur our Delhi 1...,Model Town,13.218623
853,28.821690,77.170194,houseno44/21/2village Bakhtawerpur our Delhi 1...,Model Town,13.218185
1270,28.820969,77.173190,Bakhtawarpur,Model Town,13.091734
1269,28.820787,77.173122,Bakhtawarpur,Model Town,13.072720
1290,28.857549,77.106558,Plot No.62 Opp. Radha Swami Satsang Byas,Rohini,12.977074
1153,28.861556,77.086265,Khasra 33/13/1 Gutam Colony Sfiabaadrd Narela,Rohini,12.803952
1277,28.861541,77.086228,Gali No.14 Gautam Colony SAFIABAD ROAD Narela,Rohini,12.801470
1281,28.859556,77.087649,"Sanjay Colony, Khasra No .37/10/2,Gali No 9 Ma...",Rohini,12.619178
1282,28.859523,77.087633,"Sanjay Colony .khasra No 37 /10/2,Gali No 9 Ma...",Rohini,12.615243
977,28.858000,77.093068,1961main Arya Samaj road,Rohini,12.590432


In [92]:
distance_matrix.loc[
    distance_matrix["Nearest_Distance_km"] > 1,
    ["latitude", "longitude", "Nearest_Area", "Nearest_Distance_km"]
]

,latitude,longitude,Nearest_Area,Nearest_Distance_km
2,28.571189,77.259806,Lajpat Nagar,1.676929
3,28.588991,77.253240,Lajpat Nagar,2.566167
5,28.626722,77.065972,Janakpuri,2.196949
6,28.510291,77.171653,Vasant Kunj,2.339043
7,28.522486,77.089232,IGI Airport,3.893645
...,...,...,...,...
1653,28.599304,77.064805,Dwarka,2.003065
1654,28.583150,77.180980,AIIMS,3.343053
1655,28.605720,77.103887,Janakpuri,2.388049
1657,28.592788,77.256700,Lajpat Nagar,3.089616


In [93]:
distance_matrix.shape

(1659, 30)

In [94]:
thresholds = [1, 2, 3, 5, 7, 10]

for d in thresholds:
    count = (distance_matrix["Nearest_Distance_km"] <= d).sum()
    percent = count / len(distance_matrix) * 100
    print(f"Within {d} km: {count} stations ({percent:.2f}%)")

Within 1 km: 170 stations (10.25%)
Within 2 km: 518 stations (31.22%)
Within 3 km: 825 stations (49.73%)
Within 5 km: 1372 stations (82.70%)
Within 7 km: 1575 stations (94.94%)
Within 10 km: 1630 stations (98.25%)


In [95]:
distance_matrix.to_csv(
    "../data/processed/station_area_distance_matrix.csv",
    index=False
)